# text-analysis-demo v1.0.0

A reproducible text analysis pipeline for humanities researchers. Demonstrates frequency analysis, word co-occurrence, and basic NLP on plain-text corpora.

| | |
|---|---|
| **Authors** | Tzortzina Gianni |
| **Language** | Python |
| **Keywords** | text analysis, NLP, humanities, corpus, reproducible research |
| **Repository** | [https://github.com/georginagianni/text-analysis-demo](https://github.com/georginagianni/text-analysis-demo) |
| **Dependencies** | |

- `nltk>=3.8`
- `pandas>=2.0`
- `matplotlib>=3.7`
- `wordcloud>=1.9`
- `jupyter>=1.0`

---
*Auto-generated from `codemeta.json` — no manual coding required.*

## 1. Environment check

In [ ]:
import sys
print(f'Python {sys.version}')

import importlib
missing = []
try:
    importlib.import_module('jupyter')
    print('  OK  jupyter')
except ImportError:
    missing.append('jupyter')
    print('  MISSING  jupyter')

print('All OK.' if not missing else f'Missing: {missing}')


## 2. Import dependencies

In [ ]:
import nltk
import pandas
import matplotlib
import wordcloud

print('All imports successful.')

## 3. Text cleaning utility

Run this cell first before pasting any text. It removes curly quotes and other characters that cause errors.

In [ ]:
import unicodedata

def clean_text(text):
    text = unicodedata.normalize('NFKC', text)
    replacements = {
        chr(0x2018): chr(39), chr(0x2019): chr(39),
        chr(0x201c): chr(34), chr(0x201d): chr(34),
        chr(0x2013): '-', chr(0x2014): '-',
        chr(0x2026): '...', chr(0x00a0): ' ',
        chr(0x00ad): '',
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

print('clean_text() ready.')

## 4. Dataset

No dataset URL was declared in `codemeta.json`.

Add a `contentUrl` or `distribution` field to your codemeta.json to have data loaded automatically.

Or load your own data below:

In [ ]:
# OPTION 1: Upload a file (.txt or .pdf)
# Click the upload button, select your file, then run the cell below
import ipywidgets as widgets
from IPython.display import display
import io

upload = widgets.FileUpload(accept='.txt,.pdf', multiple=False)
display(upload)
print('Select a .txt or .pdf file above, then run the next cell.')

In [ ]:
# Run this cell after uploading to load the file
if upload.value:
    # Compatible with both old and new ipywidgets API
    val = upload.value
    if isinstance(val, dict):
        file_info = list(val.values())[0]
        fname = file_info['metadata']['name']
        raw = bytes(file_info['content'])
    else:
        file_info = val[0]
        fname = file_info['name']
        raw = bytes(file_info['content'])
    if fname.endswith('.pdf'):
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'pypdf2', '-q'])
        import PyPDF2, io as _io
        reader = PyPDF2.PdfReader(_io.BytesIO(raw))
        text = ' '.join(page.extract_text() or '' for page in reader.pages)
    else:
        text = raw.decode('utf-8', errors='ignore')
    corpus = clean_text(text)
    print(f'Loaded: {fname} — {len(corpus.split())} words')
else:
    print('No file uploaded yet. Use Option 2 below to paste text instead.')

In [ ]:
# OPTION 2: Paste your text directly
# Only use this if you did NOT upload a file above
# If you uploaded a file, skip this cell entirely
if not upload.value:
    corpus = clean_text("""
Paste your text here. Any paragraph from an article, paper,
or any source relevant to your research. At least 50 words
works best for meaningful results.
""")
    print(f'Corpus loaded: {len(corpus.split())} words')
else:
    print(f'Using uploaded file — corpus already loaded: {len(corpus.split())} words')

## 5. Start exploring

The environment for **text-analysis-demo** is ready. All dependencies are installed. Add your analysis below.

Tip: use `clean_text(your_text)` on any pasted text to avoid encoding errors.

## 5b. Text analysis

This cell runs the full text analysis on your corpus and generates the word frequency chart and word cloud.

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import pandas as pd

# Download required NLTK data
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Tokenize and filter
stop_words = set(stopwords.words('english'))
tokens = word_tokenize(corpus.lower())
words = [w for w in tokens if w.isalpha() and w not in stop_words]
freq = Counter(words)

# Word frequency chart
top_words = freq.most_common(15)
df = pd.DataFrame(top_words, columns=['word', 'count'])
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(df['word'][::-1], df['count'][::-1], color='#534AB7', alpha=0.8)
ax.set_xlabel('Frequency')
ax.set_title('Top 15 words in corpus')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Word cloud
text_clean = ' '.join(words)
wc = WordCloud(width=800, height=400, background_color='white',
               colormap='viridis', max_words=80).generate(text_clean)
plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word cloud')
plt.tight_layout()
plt.savefig('wordcloud.png')
plt.show()

# Top 5 summary
freq_df = pd.DataFrame(freq.most_common(5), columns=['word', 'count'])
print('Top 5 words:')
print(freq_df.to_string())
print('Word cloud saved as wordcloud.png')

## 6. Reproducibility info

In [ ]:
import json
with open('codemeta.json') as f:
    meta = json.load(f)

print(f"Tool    : {meta.get('name','—')} {meta.get('version','')}")
print(f"License : {meta.get('license','—')}")
print(f"Repo    : {meta.get('codeRepository','—')}")
deps = meta.get('softwareRequirements',[])
print(f"Deps    : {[d.get('name',d) if isinstance(d,dict) else d for d in deps]}")
